# ClearSight AI: My Complete Learning Journey & Pipeline Construction
### By: Rajtilak Chamlagain

Hey everyone! This is my master notebook where I built, tested, and fine-tuned every single component of the **ClearSight AI** project before turning it into the final Web App (`app.py`). 

I'm documenting this so I can always look back and remember exactly **why** I wrote each line of code, how I handled the math, and how I overcame the massive hurdles of tracking people in a chaotic crowd.

---
## Phase 0: The Setup & Imports
First, we need to bring in the heavy artillery. I'm using `ultralytics` for YOLOv8 (because it's the fastest for finding humans), `deepface` (for ArcFace & RetinaFace), and standard tools like `cv2` and `numpy` for matrix math.

In [ ]:
# Bringing in the core libraries we need for the pipeline
import cv2               # OpenCV for tearing apart video frames and drawing boxes
import numpy as np       # Good old NumPy for heavy array math (like Cosine Similarity)
import torch             # PyTorch because we need that GPU juice if available!
from deepface import DeepFace  # Our wrapper for ArcFace and RetinaFace
from ultralytics import YOLO   # YOLOv8 object detection
import math
import warnings

# I hate seeing those random FutureWarnings clogging my terminal, so let's silence them.
warnings.filterwarnings('ignore')

print("All libraries successfully loaded! Ready to hunt targets. 🚀")

---
## Phase 1: Biometric Encoding (ArcFace + RetinaFace)
I tried FaceNet initially, but it completely failed when people were in the dark or turned their heads. ArcFace maps faces to a 512D math sphere (using Angular Margin Loss), which is insanely accurate.
But before ArcFace can do its magic, we need **RetinaFace** to find the 5 facial points (eyes, nose, mouth) and physically straighten the face.

In [ ]:
def get_face_embedding(img_array):
    """
    Takes a cropped image of a person, finds their face, aligns it, 
    and returns a 512-dimensional vector (their mathematical fingerprint).
    """
    try:
        # Enforce BGR to RGB conversion because OpenCV loads in BGR but DeepFace expects RGB!
        # If you forget this, everyone looks like a Smurf and the AI gets confused.
        rgb_img = cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB)
        
        # Using DeepFace to extract embeddings.
        # Model = ArcFace (best for strict identity separation)
        # Detector = RetinaFace (way better than MTCNN for tilted/dark faces)
        embedding_obj = DeepFace.represent(
            img_path=rgb_img, 
            model_name="ArcFace", 
            detector_backend="retinaface", 
            enforce_detection=False # Don't crash if a face isn't perfectly visible, just do your best!
        )
        
        # DeepFace returns a list of dictionaries (one for each face found).
        # We just grab the embedding of the most prominent face [0].
        return np.array(embedding_obj[0]['embedding'])
        
    except Exception as e:
        # If literally everything goes wrong (like passing a completely black image),
        # just return None so the pipeline doesn't violently crash.
        # print(f"Embedding failed: {e}") # Uncomment to debug
        return None

---
## Phase 2: Kinetic Localization (YOLOv8)
Now we need to find the humans. YOLOv8 is terrifyingly fast. I'm using the `yolov8n.pt` (Nano) model because I need real-time FPS on consumer laptops without melting the GPU.

In [ ]:
# Load the YOLOv8 Nano model. It's tiny but highly optimized for human detection.
# I made sure to download the weights beforehand so it doesn't try to hit the internet during a live demo.
yolo_model = YOLO("yolov8n.pt")

def detect_humans(frame):
    """
    Runs YOLOv8 on a single video frame. Returns the bounding boxes for all HUMANS.
    """
    # We only care about class '0' (which is 'person' in the COCO dataset).
    # I set conf=0.4 so we don't pick up blurry garbage in the background.
    results = yolo_model(frame, classes=[0], conf=0.4, verbose=False)
    
    boxes = []
    if len(results) > 0:
        # results[0].boxes.xyxy gives us [X_min, Y_min, X_max, Y_max]
        # I convert it to a standard python list of integers so cv2 can draw them easily later.
        for box in results[0].boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)
            boxes.append((x1, y1, x2, y2))
            
    return boxes


---
## Phase 3: The Autonomous Spectral Gap Engine
This is the part I'm most proud of. I realized that forcing a human to guess a 'Match Threshold' (like 50%) is legally dangerous. In dark videos, the real suspect might only score 40%. 
So, I wrote a math function to automatically find the massive 'gap' or 'cliff' between the highest scores and the noise.

In [ ]:
def autonomous_spectral_gap(scores_list):
    """
    Takes a list of similarity scores, sorts them, and finds the largest gap.
    Places the threshold dynamically inside that gap.
    """
    if len(scores_list) < 2:
        return 45.0 # Safe default fallback if there's only 1 person in the video
        
    # Sort scores highest to lowest
    sorted_scores = sorted(scores_list, reverse=True)
    
    max_gap = 0
    best_threshold = 45.0
    
    # Loop through and find the biggest drop-off between consecutive scores
    for i in range(len(sorted_scores) - 1):
        gap = sorted_scores[i] - sorted_scores[i+1]
        if gap > max_gap:
            max_gap = gap
            # Set the threshold exactly in the middle of the cliff!
            best_threshold = sorted_scores[i+1] + (gap / 2.0)
            
    # I put a hard floor at 30%. If the math suggests 15%, the video is just too dark/blurry
    # and we shouldn't trust it in a court of law anyway.
    return max(best_threshold, 30.0)

---
## Phase 4: Cosine Similarity Math (Trigonometry in AI)
How do we compare two 512-dimensional arrays? We use Cosine Similarity. We calculate the angle between the two vectors. If the angle is 0, they are exactly the same person. Let's write a quick function for it.

In [ ]:
def calculate_cosine_similarity(vec1, vec2):
    """
    Math 101: dot_product(A, B) / (magnitude(A) * magnitude(B))
    Returns a percentage from 0% to 100%.
    """
    # Safety check: if one of the vectors is None (face detection failed), return 0%
    if vec1 is None or vec2 is None:
        return 0.0
        
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    
    # Prevent division by zero if a vector is empty
    if norm_a == 0 or norm_b == 0:
        return 0.0
        
    similarity = dot_product / (norm_a * norm_b)
    
    # Convert standard cosine similarity (-1 to 1) into a clean percentage (0 to 100%)
    # Just clamped at 0 so we don't get negative percentages.
    percent = max(0.0, similarity * 100)
    return round(percent, 2)

---
## Conclusion & Future Implementation (Phase 5)

This notebook proves the math and logic behind the engine. I tested this exact code on various datasets (like the Rahul Gandhi walk video and the Messi crowd video) to guarantee the bounding boxes and thresholds behaved flawlessly. 

After validating everything here in Jupyter, I ported all these perfectly tested functions into `app.py`, hooked it up to **ByteTrack** for temporal memory, and wrapped it in **Streamlit** (Phase 5) to create the Zero-Lag Web Dashboard.

*(P.S. To solve the issue where the deep learning tracker occasionally drops a target due to massive occlusions, I engineered the 'Top-4 Video Split' in `app.py`. It outputs the top 4 candidate trajectories simultaneously, so a human operator can visually verify the match from multiple angles. It's a robust fallback!)*

**End of Master Notebook.** 🚀